In [ ]:
"""
====================================================================
ÉTAPE 1/3 : FUSION FINALE - Dataset GLiNER
====================================================================

Fusionne TOUS les batches en un dataset GLiNER prêt pour le training.

Output:
    C:\EnnoSmart\dataset_final\gliner_dataset_complete.json
    C:\EnnoSmart\dataset_final\stats.json
"""

import json
import re
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from collections import Counter, defaultdict

BASE_DIR = Path(r"C:\EnnoSmart")
WORK_DIR = BASE_DIR / "llm_annotation"
PROMPTS_DIR = WORK_DIR / "prompts"
RESPONSES_DIR = WORK_DIR / "responses"

OUTPUT_DIR = BASE_DIR / "dataset_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_CORE = [
    "VERROU_TECH", "METHODE_RD", "TECHNOLOGIE_RD", "EQUIPEMENT_RD",
    "COMPOSANT_TECHNIQUE", "MATERIAU_SPECIFIQUE", "DOMAINE_RD",
    "RESULTAT_RD", "OBJECTIF_RD",
]


def extract_json(content: str) -> Optional[Dict]:
    """Extrait le JSON d'une réponse Claude"""
    for pattern in [r'```json\s*(.*?)\s*```', r'```\s*(.*?)\s*```']:
        match = re.search(pattern, content, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass
    try:
        return json.loads(content.strip())
    except json.JSONDecodeError:
        pass
    start = content.find('{')
    end = content.rfind('}')
    if start != -1 and end != -1:
        try:
            return json.loads(content[start:end+1])
        except json.JSONDecodeError:
            pass
    return None


def find_correct_offset(text: str, entity_text: str, hint: int = 0):
    if not text or not entity_text:
        return None

    entity_text = entity_text.strip()
    if not entity_text:
        return None

    if isinstance(hint, int) and 0 <= hint < len(text):
        window_start = max(0, hint - 500)
        window_end = min(len(text), hint + 500)
        window = text[window_start:window_end]

        local_pos = window.find(entity_text)
        if local_pos != -1:
            pos = window_start + local_pos
            return pos, pos + len(entity_text)

    pos = text.find(entity_text)
    if pos != -1:
        return pos, pos + len(entity_text)

    return None


def validate_entity(text: str, entity: Dict) -> Optional[Dict]:
    """Valide et corrige une entité"""
    label = entity.get("label")
    ent_text = entity.get("text", "").strip()
    start = entity.get("start")
    end = entity.get("end")
    
    if label not in LABELS_CORE:
        return None
    if not ent_text or len(ent_text) < 2 or len(ent_text) > 300:
        return None
    
    # Cas 1: offsets corrects et texte correspond
    if (isinstance(start, int) and isinstance(end, int) and 
        0 <= start < end <= len(text) and 
        text[start:end] == ent_text):
        return {
            "start": start,
            "end": end,
            "label": label,
            "text": ent_text,
        }
    
    # Cas 2: chercher la bonne position
    corrected = find_correct_offset(text, ent_text)
    if corrected:
        return {
            "start": corrected[0],
            "end": corrected[1],
            "label": label,
            "text": text[corrected[0]:corrected[1]],
        }
    
    return None


def remove_overlaps(entities: List[Dict]) -> List[Dict]:
    """Supprime les chevauchements (garde la plus longue)"""
    sorted_ents = sorted(entities, key=lambda x: -(x["end"] - x["start"]))
    kept = []
    used_spans = []
    
    for ent in sorted_ents:
        s, e = ent["start"], ent["end"]
        if not any(not (e <= us or s >= ue) for us, ue in used_spans):
            kept.append(ent)
            used_spans.append((s, e))
    
    return sorted(kept, key=lambda x: x["start"])


def text_to_gliner_format(text: str, entities: List[Dict]) -> Dict:
    """
    Convertit (text, entities char-based) → format GLiNER (tokenized_text + ner)
    GLiNER attend une liste de tokens et des spans token-based.
    """
    # Tokenisation simple par whitespace + ponctuation
    tokens = []
    token_starts = []  # positions char de début de chaque token
    
    # Pattern : mots, ponctuation, autres caractères
    for match in re.finditer(r'\S+', text):
        tokens.append(match.group())
        token_starts.append(match.start())
    
    # Mapper les entités (char-based) vers token-based
    ner = []
    for ent in entities:
        char_start = ent["start"]
        char_end = ent["end"]
        
        # Trouver le token qui contient char_start
        tok_start = None
        tok_end = None
        
        for i, tok_pos in enumerate(token_starts):
            tok_end_pos = tok_pos + len(tokens[i])
            
            if tok_start is None and tok_pos <= char_start < tok_end_pos:
                tok_start = i
            
            if tok_pos < char_end <= tok_end_pos:
                tok_end = i
                break
            
            # Cas limite : entité termine pile en fin de token
            if char_end == tok_end_pos:
                tok_end = i
                break
        
        if tok_start is not None and tok_end is not None:
            ner.append([tok_start, tok_end, ent["label"]])
    
    return {
        "tokenized_text": tokens,
        "ner": ner,
    }


def main():
    print("=" * 70)
    print("ÉTAPE 1/3 : FUSION FINALE DU DATASET")
    print("=" * 70)
    
    # 1. Charger tous les batches
    response_files = sorted(RESPONSES_DIR.glob("batch_*_response.json"))
    print(f"\n{len(response_files)} fichiers de réponses trouvés")
    
    # Stats
    stats = {
        "total_chunks": 0,
        "total_entities_raw": 0,
        "total_entities_valid": 0,
        "total_hallucinations": 0,
        "total_offset_fixed": 0,
        "labels": Counter(),
        "by_project": defaultdict(lambda: {"chunks": 0, "entities": 0}),
        "batches_processed": 0,
        "batches_failed": [],
    }
    
    final_dataset = []
    final_dataset_gliner = []  # Format token-based pour GLiNER
    
    for response_file in response_files:
        match = re.match(r"batch_(\d+)_response\.json", response_file.name)
        if not match:
            continue
        
        batch_id = int(match.group(1))
        mapping_file = PROMPTS_DIR / f"batch_{batch_id:04d}_mapping.json"
        
        # Charger réponse
        with open(response_file, "r", encoding="utf-8") as f:
            response = extract_json(f.read())
        
        if not response:
            stats["batches_failed"].append(batch_id)
            continue
        
        # Charger mapping
        with open(mapping_file, "r", encoding="utf-8") as f:
            mapping = json.load(f)
        
        chunks_by_id = {c["chunk_id"]: c for c in mapping["chunks"]}
        annotations_by_id = {a["chunk_id"]: a.get("entities", []) 
                             for a in response.get("annotations", [])}
        
        # Traiter chaque chunk
        for chunk_id, chunk in chunks_by_id.items():
            text = chunk["text"]
            project_id = chunk.get("project_id", "?")
            raw_entities = annotations_by_id.get(chunk_id, [])
            
            # Valider chaque entité
            valid_entities = []
            for raw in raw_entities:
                stats["total_entities_raw"] += 1
                
                orig_start = raw.get("start")
                orig_end = raw.get("end")
                
                validated = validate_entity(text, raw)
                
                if validated:
                    if orig_start != validated["start"] or orig_end != validated["end"]:
                        stats["total_offset_fixed"] += 1
                    
                    valid_entities.append(validated)
                    stats["total_entities_valid"] += 1
                    stats["labels"][validated["label"]] += 1
                else:
                    stats["total_hallucinations"] += 1
            
            # Supprimer chevauchements
            valid_entities = remove_overlaps(valid_entities)
            
            # Format 1 : char-based (lisible)
            item_char = {
                "text": text,
                "entities": valid_entities,
                "metadata": {
                    "chunk_id": chunk_id,
                    "project_id": project_id,
                    "source_file": chunk.get("source_file", ""),
                    "batch_id": batch_id,
                }
            }
            final_dataset.append(item_char)
            
            # Format 2 : token-based (GLiNER)
            gliner_item = text_to_gliner_format(text, valid_entities)
            gliner_item["chunk_id"] = chunk_id
            gliner_item["project_id"] = project_id
            final_dataset_gliner.append(gliner_item)
            
            stats["total_chunks"] += 1
            stats["by_project"][project_id]["chunks"] += 1
            stats["by_project"][project_id]["entities"] += len(valid_entities)
        
        stats["batches_processed"] += 1
    
    # 2. Sauvegarder le dataset
    print(f"\n✅ Traitement terminé")
    print(f"   {stats['batches_processed']} batches traités")
    print(f"   {stats['total_chunks']} chunks au total")
    print(f"   {stats['total_entities_valid']} entités valides")
    
    # Format char-based (lisible, pour inspection)
    char_file = OUTPUT_DIR / "gliner_dataset_char_based.json"
    with open(char_file, "w", encoding="utf-8") as f:
        json.dump(final_dataset, f, ensure_ascii=False, indent=2)
    
    # Format GLiNER (token-based, pour training)
    gliner_file = OUTPUT_DIR / "gliner_dataset_complete.json"
    with open(gliner_file, "w", encoding="utf-8") as f:
        json.dump(final_dataset_gliner, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 Datasets sauvegardés :")
    print(f"   {char_file}")
    print(f"   {gliner_file}")
    
    # 3. Stats détaillées
    valid_rate = stats["total_entities_valid"] / max(stats["total_entities_raw"], 1) * 100
    halluc_rate = stats["total_hallucinations"] / max(stats["total_entities_raw"], 1) * 100
    
    print(f"\n" + "=" * 70)
    print(f"📊 STATISTIQUES FINALES")
    print(f"=" * 70)
    print(f"\nEntités :")
    print(f"  Total brutes      : {stats['total_entities_raw']}")
    print(f"  Valides retenues  : {stats['total_entities_valid']} ({valid_rate:.1f}%)")
    print(f"  Offsets corrigés  : {stats['total_offset_fixed']}")
    print(f"  Hallucinations    : {stats['total_hallucinations']} ({halluc_rate:.1f}%)")
    
    print(f"\nChunks :")
    chunks_with_ents = sum(1 for item in final_dataset if item["entities"])
    chunks_empty = stats["total_chunks"] - chunks_with_ents
    print(f"  Total             : {stats['total_chunks']}")
    print(f"  Avec entités      : {chunks_with_ents}")
    print(f"  Sans entités      : {chunks_empty}")
    print(f"  Moyenne ent/chunk : {stats['total_entities_valid']/max(stats['total_chunks'],1):.1f}")
    
    print(f"\nDistribution par label :")
    total_lbl = sum(stats["labels"].values())
    for label in LABELS_CORE:
        count = stats["labels"][label]
        pct = count / max(total_lbl, 1) * 100
        bar = "█" * int(pct / 2)
        print(f"  {label:<22} {count:>5} ({pct:>5.1f}%) {bar}")
    
    if label_counts := list(stats["labels"].values()):
        ratio = max(label_counts) / max(min(label_counts), 1)
        print(f"\n  Ratio max/min : {ratio:.1f}x")
    
    print(f"\nProjets :")
    for proj in sorted(stats["by_project"].keys()):
        ps = stats["by_project"][proj]
        print(f"  {proj:<15} {ps['chunks']:>4} chunks  {ps['entities']:>5} entités")
    
    # Sauvegarder stats
    stats_serializable = {
        **{k: v for k, v in stats.items() if k not in ["labels", "by_project"]},
        "labels": dict(stats["labels"]),
        "by_project": dict(stats["by_project"]),
    }
    
    stats_file = OUTPUT_DIR / "stats_final.json"
    with open(stats_file, "w", encoding="utf-8") as f:
        json.dump(stats_serializable, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 Stats sauvegardées : {stats_file}")
    
    print(f"\n" + "=" * 70)
    print(f"✅ ÉTAPE 1/3 TERMINÉE")
    print(f"=" * 70)
    print(f"\n👉 Prochaine étape : python split_dataset.py")


if __name__ == "__main__":
    main()

<>:9: SyntaxWarning: invalid escape sequence '\E'
<>:9: SyntaxWarning: invalid escape sequence '\E'
C:\Users\dell\AppData\Local\Temp\ipykernel_33712\3664698891.py:9: SyntaxWarning: invalid escape sequence '\E'
  C:\EnnoSmart\dataset_final\gliner_dataset_complete.json


ÉTAPE 1/3 : FUSION FINALE DU DATASET

122 fichiers de réponses trouvés

✅ Traitement terminé
   120 batches traités
   960 chunks au total
   9720 entités valides

💾 Datasets sauvegardés :
   C:\EnnoSmart\dataset_final\gliner_dataset_char_based.json
   C:\EnnoSmart\dataset_final\gliner_dataset_complete.json

📊 STATISTIQUES FINALES

Entités :
  Total brutes      : 10010
  Valides retenues  : 9720 (97.1%)
  Offsets corrigés  : 3955
  Hallucinations    : 290 (2.9%)

Chunks :
  Total             : 960
  Avec entités      : 785
  Sans entités      : 175
  Moyenne ent/chunk : 10.1

Distribution par label :
  VERROU_TECH             1546 ( 15.9%) ███████
  METHODE_RD              2208 ( 22.7%) ███████████
  TECHNOLOGIE_RD          1437 ( 14.8%) ███████
  EQUIPEMENT_RD            240 (  2.5%) █
  COMPOSANT_TECHNIQUE      922 (  9.5%) ████
  MATERIAU_SPECIFIQUE      348 (  3.6%) █
  DOMAINE_RD               479 (  4.9%) ██
  RESULTAT_RD             1050 ( 10.8%) █████
  OBJECTIF_RD             

In [12]:
"""
====================================================================
ÉTAPE 2/3 : SPLIT TRAIN/VAL/TEST
====================================================================

Split intelligent du dataset :
- 80% train, 10% val, 10% test
- Stratifié par label dominant de chaque chunk
- Préserve l'équilibre des labels

Output:
    C:\EnnoSmart\dataset_final\train.json
    C:\EnnoSmart\dataset_final\val.json
    C:\EnnoSmart\dataset_final\test.json
"""

import json
import random
from pathlib import Path
from collections import Counter, defaultdict

BASE_DIR = Path(r"C:\EnnoSmart")
DATASET_DIR = BASE_DIR / "dataset_final"

LABELS_CORE = [
    "VERROU_TECH", "METHODE_RD", "TECHNOLOGIE_RD", "EQUIPEMENT_RD",
    "COMPOSANT_TECHNIQUE", "MATERIAU_SPECIFIQUE", "DOMAINE_RD",
    "RESULTAT_RD", "OBJECTIF_RD",
]

# Ratios de split
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

SEED = 42


def get_dominant_label(item):
    """Retourne le label le plus fréquent dans un chunk"""
    if not item.get("ner"):
        return "EMPTY"
    
    labels = [ner[2] for ner in item["ner"]]
    return Counter(labels).most_common(1)[0][0]


def stratified_split(dataset):
    """Split stratifié par label dominant"""
    random.seed(SEED)
    
    # Grouper par label dominant
    by_dominant = defaultdict(list)
    for item in dataset:
        dominant = get_dominant_label(item)
        by_dominant[dominant].append(item)
    
    train, val, test = [], [], []
    
    for dominant, items in by_dominant.items():
        random.shuffle(items)
        
        n = len(items)
        n_train = int(n * TRAIN_RATIO)
        n_val = int(n * VAL_RATIO)
        
        train.extend(items[:n_train])
        val.extend(items[n_train:n_train + n_val])
        test.extend(items[n_train + n_val:])
    
    # Re-shuffler chaque split
    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)
    
    return train, val, test


def compute_label_distribution(dataset):
    """Calcule la distribution des labels"""
    counter = Counter()
    for item in dataset:
        for ner in item.get("ner", []):
            counter[ner[2]] += 1
    return counter


def print_distribution(name, dataset):
    """Affiche la distribution d'un dataset"""
    dist = compute_label_distribution(dataset)
    total = sum(dist.values())
    
    print(f"\n📊 {name} ({len(dataset)} chunks, {total} entités)")
    print("-" * 60)
    for label in LABELS_CORE:
        count = dist[label]
        pct = count / max(total, 1) * 100
        print(f"  {label:<22} {count:>5} ({pct:>5.1f}%)")


def main():
    print("=" * 70)
    print("ÉTAPE 2/3 : SPLIT TRAIN/VAL/TEST")
    print("=" * 70)
    
    # Charger le dataset complet
    input_file = DATASET_DIR / "gliner_dataset_complete.json"
    
    if not input_file.exists():
        print(f"\n❌ Fichier introuvable : {input_file}")
        print(f"   Lance d'abord : python 01_merge_final.py")
        return
    
    with open(input_file, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    
    print(f"\n📂 Dataset chargé : {len(dataset)} chunks")
    
    # Stats avant split
    total_dist = compute_label_distribution(dataset)
    print_distribution("DATASET COMPLET", dataset)
    
    # Split
    print(f"\n🔀 Split stratifié (seed={SEED})")
    print(f"   Train : {TRAIN_RATIO*100:.0f}%")
    print(f"   Val   : {VAL_RATIO*100:.0f}%")
    print(f"   Test  : {TEST_RATIO*100:.0f}%")
    
    train, val, test = stratified_split(dataset)
    
    # Stats par split
    print_distribution("TRAIN", train)
    print_distribution("VAL", val)
    print_distribution("TEST", test)
    
    # Sauvegarder
    splits = {
        "train.json": train,
        "val.json": val,
        "test.json": test,
    }
    
    print(f"\n💾 Sauvegarde des splits :")
    for filename, data in splits.items():
        path = DATASET_DIR / filename
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"   {path} ({len(data)} chunks)")
    
    # Sauvegarder les stats du split
    split_stats = {
        "seed": SEED,
        "ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
        "sizes": {
            "train": len(train),
            "val": len(val),
            "test": len(test),
            "total": len(dataset),
        },
        "distributions": {
            "train": dict(compute_label_distribution(train)),
            "val": dict(compute_label_distribution(val)),
            "test": dict(compute_label_distribution(test)),
        },
    }
    
    stats_file = DATASET_DIR / "split_stats.json"
    with open(stats_file, "w", encoding="utf-8") as f:
        json.dump(split_stats, f, ensure_ascii=False, indent=2)
    
    print(f"   {stats_file}")
    
    print(f"\n" + "=" * 70)
    print(f"✅ ÉTAPE 2/3 TERMINÉE")
    print(f"=" * 70)
    print(f"\n👉 Prochaine étape : python 03_train_gliner.py")


if __name__ == "__main__":
    main()

<>:12: SyntaxWarning: invalid escape sequence '\E'
<>:12: SyntaxWarning: invalid escape sequence '\E'
C:\Users\dell\AppData\Local\Temp\ipykernel_33712\792405791.py:12: SyntaxWarning: invalid escape sequence '\E'
  C:\EnnoSmart\dataset_final\train.json


ÉTAPE 2/3 : SPLIT TRAIN/VAL/TEST

📂 Dataset chargé : 960 chunks

📊 DATASET COMPLET (960 chunks, 9480 entités)
------------------------------------------------------------
  VERROU_TECH             1529 ( 16.1%)
  METHODE_RD              2178 ( 23.0%)
  TECHNOLOGIE_RD          1383 ( 14.6%)
  EQUIPEMENT_RD            220 (  2.3%)
  COMPOSANT_TECHNIQUE      876 (  9.2%)
  MATERIAU_SPECIFIQUE      324 (  3.4%)
  DOMAINE_RD               461 (  4.9%)
  RESULTAT_RD             1035 ( 10.9%)
  OBJECTIF_RD             1474 ( 15.5%)

🔀 Split stratifié (seed=42)
   Train : 80%
   Val   : 10%
   Test  : 10%

📊 TRAIN (766 chunks, 7547 entités)
------------------------------------------------------------
  VERROU_TECH             1200 ( 15.9%)
  METHODE_RD              1754 ( 23.2%)
  TECHNOLOGIE_RD          1129 ( 15.0%)
  EQUIPEMENT_RD            158 (  2.1%)
  COMPOSANT_TECHNIQUE      693 (  9.2%)
  MATERIAU_SPECIFIQUE      258 (  3.4%)
  DOMAINE_RD               366 (  4.8%)
  RESULTAT_RD     

In [16]:
"""
====================================================================
TRAINING NER R&D/CIR - XLM-RoBERTa BIO
====================================================================

Objectif :
- Remplacer GLiNER par un modèle token-classification plus stable.
- Support multilingue : français, anglais, arabe, etc.
- Utilise les fichiers déjà générés :
    C:\\EnnoSmart\\dataset_final\\train.json
    C:\\EnnoSmart\\dataset_final\\val.json
    C:\\EnnoSmart\\dataset_final\\test.json

Format attendu en entrée :
{
  "tokenized_text": ["token1", "token2", ...],
  "ner": [[start_token, end_token, "LABEL"], ...]
}
"""

import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Any

import torch
from datasets import Dataset
from seqeval.metrics import precision_score, recall_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)


# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = Path(r"C:\EnnoSmart")
DATASET_DIR = BASE_DIR / "dataset_final"

MODEL_NAME = "xlm-roberta-base"

OUTPUT_DIR = BASE_DIR / "models" / "xlmr_cir_ner_quick_test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
set_seed(SEED)

# ============================================================================
# MODE TEST RAPIDE
# ============================================================================

QUICK_TEST = True

if QUICK_TEST:
    TRAIN_SIZE = 100
    VAL_SIZE = 30
    NUM_EPOCHS = 1
    BATCH_SIZE = 4
    EVAL_STEPS = 20
    SAVE_STEPS = 20
else:
    TRAIN_SIZE = None
    VAL_SIZE = None
    NUM_EPOCHS = 5
    BATCH_SIZE = 8
    EVAL_STEPS = 100
    SAVE_STEPS = 100

MAX_LENGTH = 256
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1


# ============================================================================
# LABELS
# ============================================================================

LABELS_CORE = [
    "VERROU_TECH",
    "METHODE_RD",
    "TECHNOLOGIE_RD",
    "EQUIPEMENT_RD",
    "COMPOSANT_TECHNIQUE",
    "MATERIAU_SPECIFIQUE",
    "DOMAINE_RD",
    "RESULTAT_RD",
    "OBJECTIF_RD",
]

LABEL_LIST = ["O"]

for label in LABELS_CORE:
    LABEL_LIST.append(f"B-{label}")
    LABEL_LIST.append(f"I-{label}")

label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}


# ============================================================================
# DATA LOADING
# ============================================================================

def load_json(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def has_entities(item: Dict[str, Any]) -> bool:
    tokens = item.get("tokenized_text", [])
    ner = item.get("ner", [])

    if not tokens or not ner:
        return False

    for span in ner:
        if len(span) >= 3:
            start, end, label = span[0], span[1], span[2]

            if (
                isinstance(start, int)
                and isinstance(end, int)
                and 0 <= start <= end < len(tokens)
                and label in LABELS_CORE
            ):
                return True

    return False


def convert_to_bio(item: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convertit tokenized_text + ner spans vers tokens + labels BIO.
    """
    tokens = item.get("tokenized_text", [])
    ner = item.get("ner", [])

    bio_labels = ["O"] * len(tokens)

    for span in ner:
        if len(span) < 3:
            continue

        start, end, label = span[0], span[1], span[2]

        if label not in LABELS_CORE:
            continue

        if not isinstance(start, int) or not isinstance(end, int):
            continue

        if start < 0 or end >= len(tokens) or start > end:
            continue

        bio_labels[start] = f"B-{label}"

        for i in range(start + 1, end + 1):
            bio_labels[i] = f"I-{label}"

    return {
        "tokens": tokens,
        "ner_tags": [label2id[x] for x in bio_labels],
        "chunk_id": item.get("chunk_id", ""),
        "project_id": item.get("project_id", ""),
    }


def prepare_dataset(data: List[Dict[str, Any]], name: str, limit=None) -> Dataset:
    before = len(data)

    # Pour le test rapide, on garde seulement les chunks avec entités.
    # Après validation, on pourra réintégrer les chunks sans entités.
    data = [x for x in data if has_entities(x)]

    if limit is not None:
        data = data[:limit]

    converted = [convert_to_bio(x) for x in data]

    print(f"\n{name}:")
    print(f"   Avant filtrage : {before}")
    print(f"   Après filtrage : {len(data)}")
    print(f"   Convertis BIO  : {len(converted)}")

    lengths = [len(x["tokens"]) for x in converted]

    if lengths:
        print(f"   Longueur min   : {min(lengths)}")
        print(f"   Longueur max   : {max(lengths)}")
        print(f"   Longueur moy   : {sum(lengths) / len(lengths):.1f}")

    return Dataset.from_list(converted)


# ============================================================================
# TOKENIZATION + LABEL ALIGNMENT
# ============================================================================

def tokenize_and_align_labels(examples, tokenizer):
    """
    Aligne les labels BIO avec les sous-tokens XLM-RoBERTa.

    Règle :
    - Premier sous-token d'un mot : garde le label BIO.
    - Sous-tokens suivants : -100, ignorés dans la loss.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)

            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])

            else:
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels

    return tokenized_inputs


# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            current_preds.append(id2label[int(pred_id)])
            current_labels.append(id2label[int(label_id)])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 70)
    print("TRAINING XLM-RoBERTa NER R&D/CIR")
    print("=" * 70)

    train_path = DATASET_DIR / "train.json"
    val_path = DATASET_DIR / "val.json"

    train_raw = load_json(train_path)
    val_raw = load_json(val_path)

    train_dataset = prepare_dataset(
        train_raw,
        name="TRAIN",
        limit=TRAIN_SIZE if QUICK_TEST else None,
    )

    val_dataset = prepare_dataset(
        val_raw,
        name="VAL",
        limit=VAL_SIZE if QUICK_TEST else None,
    )

    print("\nLabels BIO :")
    for i, label in enumerate(LABEL_LIST):
        print(f"   {i:02d} -> {label}")

    print(f"\nChargement tokenizer : {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("\nTokenisation + alignement labels...")

    train_tokenized = train_dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    val_tokenized = val_dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=val_dataset.column_names,
    )

    print(f"\nChargement modèle : {MODEL_NAME}")

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_LIST),
        id2label=id2label,
        label2id=label2id,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    print("\nDevice :")
    print(f"   CUDA disponible : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"   GPU  : {torch.cuda.get_device_name(0)}")
        print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    print("\nHyperparamètres :")
    print(f"   QUICK_TEST      : {QUICK_TEST}")
    print(f"   Epochs          : {NUM_EPOCHS}")
    print(f"   Batch size      : {BATCH_SIZE}")
    print(f"   Max length      : {MAX_LENGTH}")
    print(f"   Learning rate   : {LEARNING_RATE}")
    print(f"   Output dir      : {OUTPUT_DIR}")

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),

        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,

        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,

        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        logging_strategy="steps",
        logging_steps=10,

        fp16=torch.cuda.is_available(),

        report_to="none",
        remove_unused_columns=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,

        # Correction pour ta version de transformers :
        # tokenizer=tokenizer provoque TypeError.
        processing_class=tokenizer,

        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("\nDÉMARRAGE TRAINING XLM-RoBERTa")
    print("=" * 70)

    trainer.train()

    print("\nÉvaluation finale VAL")
    metrics = trainer.evaluate()
    print(metrics)

    print("\nSauvegarde modèle final...")
    final_dir = OUTPUT_DIR / "final"

    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    print("\nTERMINÉ")
    print(f"Modèle sauvegardé : {final_dir}")


if __name__ == "__main__":
    main()

TRAINING XLM-RoBERTa NER R&D/CIR

TRAIN:
   Avant filtrage : 766
   Après filtrage : 100
   Convertis BIO  : 100
   Longueur min   : 77
   Longueur max   : 894
   Longueur moy   : 290.2

VAL:
   Avant filtrage : 92
   Après filtrage : 30
   Convertis BIO  : 30
   Longueur min   : 50
   Longueur max   : 958
   Longueur moy   : 283.7

Labels BIO :
   00 -> O
   01 -> B-VERROU_TECH
   02 -> I-VERROU_TECH
   03 -> B-METHODE_RD
   04 -> I-METHODE_RD
   05 -> B-TECHNOLOGIE_RD
   06 -> I-TECHNOLOGIE_RD
   07 -> B-EQUIPEMENT_RD
   08 -> I-EQUIPEMENT_RD
   09 -> B-COMPOSANT_TECHNIQUE
   10 -> I-COMPOSANT_TECHNIQUE
   11 -> B-MATERIAU_SPECIFIQUE
   12 -> I-MATERIAU_SPECIFIQUE
   13 -> B-DOMAINE_RD
   14 -> I-DOMAINE_RD
   15 -> B-RESULTAT_RD
   16 -> I-RESULTAT_RD
   17 -> B-OBJECTIF_RD
   18 -> I-OBJECTIF_RD

Chargement tokenizer : xlm-roberta-base

Tokenisation + alignement labels...


Map: 100%|██████████| 30/30 [00:00<00:00, 1000.02 examples/s]



Chargement modèle : xlm-roberta-base


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 703.41it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5


Device :
   CUDA disponible : True
   GPU  : NVIDIA RTX 1000 Ada Generation Laptop GPU
   VRAM : 6.4 GB

Hyperparamètres :
   QUICK_TEST      : True
   Epochs          : 1
   Batch size      : 4
   Max length      : 256
   Learning rate   : 2e-05
   Output dir      : C:\EnnoSmart\models\xlmr_cir_ner_quick_test

DÉMARRAGE TRAINING XLM-RoBERTa


...


Évaluation finale VAL


c:\EnnoSmart\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 1.3431297540664673, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 0.3763, 'eval_samples_per_second': 79.724, 'eval_steps_per_second': 21.26, 'epoch': 1.0}

Sauvegarde modèle final...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]



TERMINÉ
Modèle sauvegardé : C:\EnnoSmart\models\xlmr_cir_ner_quick_test\final


In [ ]:
r"""
====================================================================
TRAINING NER R&D/CIR - mBERT BIO - VERSION POC PROPRE
====================================================================

Objectif :
- Entraîner un modèle NER multilingue spécialisé R&D/CIR.
- Modèle : bert-base-multilingual-cased.
- Entrée :
    C:\\EnnoSmart\\dataset_final\\train.json
    C:\\EnnoSmart\\dataset_final\\val.json
    C:\\EnnoSmart\\dataset_final\\test.json

Points importants :
- Utilise une fenêtre glissante pour ne pas perdre les chunks longs.
- Utilise des class weights pour éviter que le modèle prédise seulement O.
- Utilise early stopping pour éviter l'overfitting.
- Ignore les labels non sélectionnés au lieu de les transformer en O.
"""

import json
import math
import numpy as np
from pathlib import Path
from typing import List, Dict, Any

import torch
import torch.nn as nn

from datasets import Dataset
from seqeval.metrics import precision_score, recall_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)


# ============================================================================
# CONFIGURATION GÉNÉRALE
# ============================================================================

BASE_DIR = Path(r"C:\EnnoSmart")
DATASET_DIR = BASE_DIR / "dataset_final"

MODEL_NAME = "bert-base-multilingual-cased"

OUTPUT_DIR = BASE_DIR / "models" / "mbert_rd_cir_poc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
set_seed(SEED)


# ============================================================================
# MODE TRAINING
# ============================================================================

# Pour un POC fort, je conseille 5 labels au début.
# C'est plus stable et plus lisible pour l'encadrant.
USE_5_LABELS_POC = True

if USE_5_LABELS_POC:
    TRAIN_LABELS = [
        "DOMAINE_RD",
        "VERROU_TECH",
        "TECHNOLOGIE_RD",
        "METHODE_RD",
        "OBJECTIF_RD",
    ]
else:
    TRAIN_LABELS = [
        "VERROU_TECH",
        "METHODE_RD",
        "TECHNOLOGIE_RD",
        "EQUIPEMENT_RD",
        "COMPOSANT_TECHNIQUE",
        "MATERIAU_SPECIFIQUE",
        "DOMAINE_RD",
        "RESULTAT_RD",
        "OBJECTIF_RD",
    ]


# ============================================================================
# HYPERPARAMÈTRES ANTI OVERFITTING / UNDERFITTING
# ============================================================================

MAX_LENGTH = 256
STRIDE = 64

NUM_EPOCHS = 8
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

EVAL_STEPS = 50
SAVE_STEPS = 50
LOGGING_STEPS = 10

EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

FP16 = torch.cuda.is_available()


# ============================================================================
# LABELS BIO
# ============================================================================

LABEL_LIST = ["O"]

for label in TRAIN_LABELS:
    LABEL_LIST.append(f"B-{label}")
    LABEL_LIST.append(f"I-{label}")

label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}


def b_to_i_label(label_id: int) -> int:
    """
    Si label_id correspond à B-XXX, retourne I-XXX.
    Sinon retourne label_id.
    Structure LABEL_LIST :
      O = 0
      B-label = impair
      I-label = impair + 1
    """
    if label_id > 0 and label_id % 2 == 1:
        return label_id + 1
    return label_id


# ============================================================================
# DATA LOADING
# ============================================================================

def load_json(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def has_valid_tokens(item: Dict[str, Any]) -> bool:
    return bool(item.get("tokenized_text"))


def convert_to_word_labels(item: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convertit le format :
        tokenized_text + ner spans
    vers :
        tokens + word_labels

    Important :
    - Les labels sélectionnés deviennent BIO.
    - Les labels non sélectionnés deviennent -100 pour être ignorés,
      et non pas O. Cela évite d'apprendre à tort que ces spans sont du bruit.
    """
    tokens = item.get("tokenized_text", [])
    ner = item.get("ner", [])

    word_labels = [label2id["O"]] * len(tokens)

    for span in ner:
        if len(span) < 3:
            continue

        start, end, label = span[0], span[1], span[2]

        if not isinstance(start, int) or not isinstance(end, int):
            continue

        if start < 0 or end >= len(tokens) or start > end:
            continue

        if label not in TRAIN_LABELS:
            # On ignore ces tokens pendant la loss.
            for i in range(start, end + 1):
                word_labels[i] = -100
            continue

        word_labels[start] = label2id[f"B-{label}"]

        for i in range(start + 1, end + 1):
            word_labels[i] = label2id[f"I-{label}"]

    return {
        "tokens": tokens,
        "word_labels": word_labels,
        "chunk_id": item.get("chunk_id", ""),
        "project_id": item.get("project_id", ""),
    }


def prepare_dataset(data: List[Dict[str, Any]], name: str) -> Dataset:
    before = len(data)

    data = [x for x in data if has_valid_tokens(x)]
    converted = [convert_to_word_labels(x) for x in data]

    lengths = [len(x["tokens"]) for x in converted]
    entity_tokens = sum(
        1 for x in converted for y in x["word_labels"] if y not in [0, -100]
    )
    ignored_tokens = sum(
        1 for x in converted for y in x["word_labels"] if y == -100
    )

    print(f"\n{name}:")
    print(f"   Avant filtrage      : {before}")
    print(f"   Après filtrage      : {len(converted)}")
    print(f"   Tokens entités      : {entity_tokens}")
    print(f"   Tokens ignorés      : {ignored_tokens}")

    if lengths:
        print(f"   Longueur min        : {min(lengths)}")
        print(f"   Longueur max        : {max(lengths)}")
        print(f"   Longueur moyenne    : {sum(lengths) / len(lengths):.1f}")

    return Dataset.from_list(converted)


# ============================================================================
# TOKENIZATION AVEC FENÊTRE GLISSANTE
# ============================================================================

def tokenize_and_align_labels(examples, tokenizer):
    """
    Tokenisation avec overflow / sliding window.

    Pourquoi ?
    - Tes chunks peuvent dépasser 900 tokens.
    - Si on tronque simplement à 256, on perd beaucoup d'entités.
    - Ici, on crée plusieurs fenêtres de 256 avec stride=64.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        padding=False,
    )

    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    aligned_labels = []

    for window_index in range(len(tokenized_inputs["input_ids"])):
        sample_index = sample_mapping[window_index]
        word_ids = tokenized_inputs.word_ids(batch_index=window_index)

        original_word_labels = examples["word_labels"][sample_index]

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)

            else:
                original_label = original_word_labels[word_id]

                if original_label == -100:
                    label_ids.append(-100)

                elif word_id != previous_word_id:
                    label_ids.append(original_label)

                else:
                    # Pour les sous-tokens, on propage le label.
                    # Si c'est B-XXX, on transforme en I-XXX.
                    label_ids.append(b_to_i_label(original_label))

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels

    return tokenized_inputs


# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            current_preds.append(id2label[int(pred_id)])
            current_labels.append(id2label[int(label_id)])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    precision = precision_score(true_labels, true_predictions, zero_division=0)
    recall = recall_score(true_labels, true_predictions, zero_division=0)
    f1 = f1_score(true_labels, true_predictions, zero_division=0)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# ============================================================================
# CLASS WEIGHTS
# ============================================================================

def compute_class_weights(tokenized_dataset, num_labels: int) -> torch.Tensor:
    """
    Calcule des poids de classes pour réduire le problème :
        le modèle prédit trop souvent O.

    Méthode :
    - On compte les labels hors -100.
    - On applique un inverse sqrt frequency.
    - On réduit volontairement le poids de O.
    """
    counts = np.zeros(num_labels, dtype=np.float64)

    for item in tokenized_dataset:
        for label_id in item["labels"]:
            if label_id != -100:
                counts[int(label_id)] += 1

    counts = np.maximum(counts, 1.0)

    total = counts.sum()
    freqs = counts / total

    weights = 1.0 / np.sqrt(freqs)
    weights = weights / weights.mean()

    # La classe O domine toujours énormément.
    # On baisse son poids pour forcer le modèle à apprendre les entités.
    weights[label2id["O"]] = min(weights[label2id["O"]], 0.25)

    # Clamp pour éviter des poids extrêmes.
    weights = np.clip(weights, 0.20, 6.0)

    print("\nClass weights:")
    for i, w in enumerate(weights):
        print(f"   {id2label[i]:<25} {w:.3f} | count={int(counts[i])}")

    return torch.tensor(weights, dtype=torch.float)


# ============================================================================
# CUSTOM TRAINER AVEC LOSS PONDÉRÉE
# ============================================================================

class WeightedNERTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None

        loss_fct = nn.CrossEntropyLoss(
            weight=weights,
            ignore_index=-100,
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )

        return (loss, outputs) if return_outputs else loss


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 70)
    print("TRAINING mBERT R&D/CIR - VERSION POC")
    print("=" * 70)

    print("\nLabels utilisés :")
    for label in TRAIN_LABELS:
        print(f"   - {label}")

    print("\nLabels BIO :")
    for i, label in enumerate(LABEL_LIST):
        print(f"   {i:02d} -> {label}")

    train_raw = load_json(DATASET_DIR / "train.json")
    val_raw = load_json(DATASET_DIR / "val.json")
    test_raw = load_json(DATASET_DIR / "test.json")

    train_dataset = prepare_dataset(train_raw, "TRAIN")
    val_dataset = prepare_dataset(val_raw, "VAL")
    test_dataset = prepare_dataset(test_raw, "TEST")

    print(f"\nChargement tokenizer : {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("\nTokenisation + fenêtres glissantes...")

    train_tokenized = train_dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    val_tokenized = val_dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=val_dataset.column_names,
    )

    test_tokenized = test_dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=test_dataset.column_names,
    )

    print("\nTaille après fenêtres glissantes :")
    print(f"   Train windows : {len(train_tokenized)}")
    print(f"   Val windows   : {len(val_tokenized)}")
    print(f"   Test windows  : {len(test_tokenized)}")

    print(f"\nChargement modèle : {MODEL_NAME}")

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_LIST),
        id2label=id2label,
        label2id=label2id,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    class_weights = compute_class_weights(
        train_tokenized,
        num_labels=len(LABEL_LIST),
    )

    print("\nDevice :")
    print(f"   CUDA disponible : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"   GPU  : {torch.cuda.get_device_name(0)}")
        print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    print("\nHyperparamètres :")
    print(f"   Model                   : {MODEL_NAME}")
    print(f"   Epochs                  : {NUM_EPOCHS}")
    print(f"   Batch size              : {BATCH_SIZE}")
    print(f"   Gradient accumulation   : {GRADIENT_ACCUMULATION_STEPS}")
    print(f"   Effective batch         : {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"   Max length              : {MAX_LENGTH}")
    print(f"   Stride                  : {STRIDE}")
    print(f"   Learning rate           : {LEARNING_RATE}")
    print(f"   Weight decay            : {WEIGHT_DECAY}")
    print(f"   Early stopping patience : {EARLY_STOPPING_PATIENCE}")
    print(f"   Output dir              : {OUTPUT_DIR}")

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),

        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,

        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,

        fp16=FP16,

        report_to="none",
        remove_unused_columns=True,
    )

    trainer = WeightedNERTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,

        # Important pour ta version transformers récente
        processing_class=tokenizer,

        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            )
        ],
    )

    print("\nDÉMARRAGE TRAINING mBERT R&D/CIR")
    print("=" * 70)

    trainer.train()

    print("\nÉvaluation finale VAL")
    val_metrics = trainer.evaluate(eval_dataset=val_tokenized)
    print(val_metrics)

    print("\nÉvaluation finale TEST")
    test_metrics = trainer.evaluate(eval_dataset=test_tokenized)
    print(test_metrics)

    print("\nSauvegarde modèle final...")
    final_dir = OUTPUT_DIR / "final"

    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    metrics_file = OUTPUT_DIR / "final_metrics.json"
    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(
            {
                "model_name": MODEL_NAME,
                "train_labels": TRAIN_LABELS,
                "label_list": LABEL_LIST,
                "val_metrics": val_metrics,
                "test_metrics": test_metrics,
                "hyperparameters": {
                    "num_epochs": NUM_EPOCHS,
                    "batch_size": BATCH_SIZE,
                    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
                    "max_length": MAX_LENGTH,
                    "stride": STRIDE,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "warmup_ratio": WARMUP_RATIO,
                    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                },
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("\nTERMINÉ")
    print(f"Modèle sauvegardé : {final_dir}")
    print(f"Métriques          : {metrics_file}")


if __name__ == "__main__":
    main()

QUICK TEST BERT MULTILINGUE NER R&D/CIR

TRAIN:
   Avant filtrage : 766
   Après filtrage : 300
   Convertis BIO  : 300
   Longueur min   : 51
   Longueur max   : 958
   Longueur moy   : 282.7

VAL:
   Avant filtrage : 92
   Après filtrage : 60
   Convertis BIO  : 60
   Longueur min   : 50
   Longueur max   : 958
   Longueur moy   : 284.8

Labels BIO :
   00 -> O
   01 -> B-VERROU_TECH
   02 -> I-VERROU_TECH
   03 -> B-METHODE_RD
   04 -> I-METHODE_RD
   05 -> B-TECHNOLOGIE_RD
   06 -> I-TECHNOLOGIE_RD
   07 -> B-EQUIPEMENT_RD
   08 -> I-EQUIPEMENT_RD
   09 -> B-COMPOSANT_TECHNIQUE
   10 -> I-COMPOSANT_TECHNIQUE
   11 -> B-MATERIAU_SPECIFIQUE
   12 -> I-MATERIAU_SPECIFIQUE
   13 -> B-DOMAINE_RD
   14 -> I-DOMAINE_RD
   15 -> B-RESULTAT_RD
   16 -> I-RESULTAT_RD
   17 -> B-OBJECTIF_RD
   18 -> I-OBJECTIF_RD

Chargement tokenizer : bert-base-multilingual-cased


c:\EnnoSmart\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dell\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



Tokenisation + alignement labels...


Map: 100%|██████████| 60/60 [00:00<00:00, 1392.82 examples/s]



Chargement modèle : bert-base-multilingual-cased


...

Step,Training Loss,Validation Loss,Precision,Recall,F1
50,1.260517,1.207219,0.000000,0.000000,0.000000
100,1.166440,1.075743,0.008130,0.002212,0.003478
150,0.997141,1.020328,0.045181,0.016593,0.024272
200,0.868204,0.968871,0.043956,0.022124,0.029433



Évaluation finale VAL


{'eval_loss': 0.956725537776947, 'eval_precision': 0.050505050505050504, 'eval_recall': 0.02765486725663717, 'eval_f1': 0.035739814152966405, 'eval_runtime': 0.4188, 'eval_samples_per_second': 143.252, 'eval_steps_per_second': 35.813, 'epoch': 3.0}

Sauvegarde modèle test final...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


TERMINÉ
Modèle sauvegardé : C:\EnnoSmart\models\mbert_cir_ner_quick_test\final


In [14]:
! pip install datasets seqeval

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16282 sha256=ffa088ab482bc09ae5e83e62d67470fed8d4ba3376bdbe3287fa37f7028e63f1
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\5f\b8\73\0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [1]:
"""
====================================================================
DIAGNOSTIC : Pourquoi la loss explose ?
====================================================================

Vérifie le format des données et l'alignement labels/tokens.
"""

import json
from pathlib import Path
from collections import Counter

DATASET_DIR = Path(r"C:\EnnoSmart\dataset_final")


def main():
    print("=" * 70)
    print("DIAGNOSTIC FORMAT DES DONNÉES")
    print("=" * 70)
    
    train_file = DATASET_DIR / "train.json"
    
    with open(train_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    print(f"\n📂 {len(data)} chunks chargés")
    
    # 1. Vérifier la structure
    print("\n" + "=" * 70)
    print("1. STRUCTURE DU PREMIER ITEM")
    print("=" * 70)
    
    first = data[0]
    print(f"\nClés du premier item : {list(first.keys())}")
    print(f"\nExemple :")
    print(f"  tokenized_text (10 premiers) : {first['tokenized_text'][:10]}")
    print(f"  Type tokenized_text : {type(first['tokenized_text'])}")
    print(f"  Nombre de tokens : {len(first['tokenized_text'])}")
    print(f"\n  ner (5 premiers) : {first['ner'][:5]}")
    print(f"  Type ner : {type(first['ner'])}")
    print(f"  Nombre d'entités : {len(first['ner'])}")
    
    # 2. Vérifier le format ner
    print("\n" + "=" * 70)
    print("2. FORMAT DES ENTITÉS (NER)")
    print("=" * 70)
    
    if first['ner']:
        first_ner = first['ner'][0]
        print(f"\nPremière entité : {first_ner}")
        print(f"Type : {type(first_ner)}")
        print(f"Longueur : {len(first_ner)}")
        
        if len(first_ner) == 3:
            print(f"  [0] start_token = {first_ner[0]} (type: {type(first_ner[0]).__name__})")
            print(f"  [1] end_token   = {first_ner[1]} (type: {type(first_ner[1]).__name__})")
            print(f"  [2] label       = {first_ner[2]} (type: {type(first_ner[2]).__name__})")
            
            # Reconstruire le texte de l'entité
            tokens = first['tokenized_text']
            s, e = first_ner[0], first_ner[1]
            if s < len(tokens) and e < len(tokens):
                entity_text = " ".join(tokens[s:e+1])
                print(f"  Texte reconstruit : '{entity_text}'")
    
    # 3. Vérifier la validité de toutes les entités
    print("\n" + "=" * 70)
    print("3. VALIDATION DE TOUTES LES ENTITÉS")
    print("=" * 70)
    
    total_entities = 0
    invalid_entities = 0
    out_of_bounds = 0
    inverted = 0  # end < start
    label_counter = Counter()
    
    for i, item in enumerate(data):
        tokens = item.get("tokenized_text", [])
        n_tokens = len(tokens)
        
        for ner in item.get("ner", []):
            total_entities += 1
            
            if len(ner) != 3:
                invalid_entities += 1
                continue
            
            s, e, lbl = ner[0], ner[1], ner[2]
            
            # Vérifier types
            if not isinstance(s, int) or not isinstance(e, int):
                invalid_entities += 1
                continue
            
            # Vérifier bornes
            if s < 0 or e < 0 or s >= n_tokens or e >= n_tokens:
                out_of_bounds += 1
                continue
            
            # Vérifier ordre
            if e < s:
                inverted += 1
                continue
            
            label_counter[lbl] += 1
    
    print(f"\nTotal entités     : {total_entities}")
    print(f"Invalides         : {invalid_entities}")
    print(f"Hors bornes       : {out_of_bounds}")
    print(f"Start > End       : {inverted}")
    print(f"\nDistribution labels :")
    for lbl, count in label_counter.most_common():
        print(f"  {lbl:<25} : {count}")
    
    # 4. Statistiques des longueurs
    print("\n" + "=" * 70)
    print("4. STATISTIQUES DES TOKENS")
    print("=" * 70)
    
    token_lengths = [len(item.get("tokenized_text", [])) for item in data]
    
    print(f"\nMin tokens   : {min(token_lengths)}")
    print(f"Max tokens   : {max(token_lengths)}")
    print(f"Moyenne      : {sum(token_lengths)/len(token_lengths):.0f}")
    print(f"Médiane      : {sorted(token_lengths)[len(token_lengths)//2]}")
    
    # 5. Chunks vides
    empty = sum(1 for item in data if not item.get("ner"))
    print(f"\nChunks sans entités : {empty} ({empty/len(data)*100:.1f}%)")
    
    # 6. Échantillon visuel
    print("\n" + "=" * 70)
    print("5. ÉCHANTILLON VISUEL (3 premiers chunks)")
    print("=" * 70)
    
    for i in range(min(3, len(data))):
        item = data[i]
        tokens = item["tokenized_text"]
        print(f"\n--- Chunk {i} ---")
        print(f"Tokens (premiers 30) : {tokens[:30]}")
        print(f"Total tokens : {len(tokens)}")
        print(f"Entités ({len(item['ner'])}) :")
        for ner in item['ner'][:5]:
            s, e, lbl = ner[0], ner[1], ner[2]
            if s < len(tokens) and e < len(tokens):
                entity_text = " ".join(tokens[s:e+1])
                print(f"  [{s}-{e}] {lbl}: '{entity_text}'")
        if len(item['ner']) > 5:
            print(f"  ... +{len(item['ner'])-5} autres")
    
    # VERDICT
    print("\n" + "=" * 70)
    print("VERDICT")
    print("=" * 70)
    
    if out_of_bounds > 0:
        print(f"\n❌ PROBLÈME : {out_of_bounds} entités hors bornes !")
        print(f"   Probablement la cause de la loss qui explose.")
    
    if invalid_entities > 0:
        print(f"\n❌ PROBLÈME : {invalid_entities} entités au format invalide !")
    
    if inverted > 0:
        print(f"\n⚠️ {inverted} entités avec end < start")
    
    if out_of_bounds == 0 and invalid_entities == 0 and inverted == 0:
        print(f"\n✅ Le format des données semble OK")
        print(f"   Le problème est ailleurs (peut-être les params du collator)")


if __name__ == "__main__":
    main()

DIAGNOSTIC FORMAT DES DONNÉES

📂 766 chunks chargés

1. STRUCTURE DU PREMIER ITEM

Clés du premier item : ['tokenized_text', 'ner', 'chunk_id', 'project_id']

Exemple :
  tokenized_text (10 premiers) : ['[SECTION', ':', 'Conclusion', 'et', 'contribution', 'scientifique,', 'technique', 'ou', 'technologique]', 'Les']
  Type tokenized_text : <class 'list'>
  Nombre de tokens : 344

  ner (5 premiers) : [[27, 44, 'OBJECTIF_RD'], [61, 80, 'TECHNOLOGIE_RD'], [100, 123, 'VERROU_TECH'], [128, 130, 'VERROU_TECH'], [163, 183, 'TECHNOLOGIE_RD']]
  Type ner : <class 'list'>
  Nombre d'entités : 10

2. FORMAT DES ENTITÉS (NER)

Première entité : [27, 44, 'OBJECTIF_RD']
Type : <class 'list'>
Longueur : 3
  [0] start_token = 27 (type: int)
  [1] end_token   = 44 (type: int)
  [2] label       = OBJECTIF_RD (type: str)
  Texte reconstruit : 'développer un banc de test permettant la mise en œuvre et l’étude de différents systèmes de guidage d’engins'

3. VALIDATION DE TOUTES LES ENTITÉS

Total entités  

In [ ]:
"""
====================================================================
VISUALISATION DES COURBES DE TRAINING
====================================================================

À lancer PENDANT ou APRÈS le training pour voir les courbes.

Usage:
    python plot_training_curves.py
"""

import csv
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE_DIR = Path(r"C:\EnnoSmart")
MODEL_DIR = BASE_DIR / "models" / "gliner_cir_v2"
CSV_PATH = MODEL_DIR / "training_log.csv"


def load_history(csv_path):
    """Charge l'historique depuis le CSV"""
    history = []
    
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            entry = {}
            for k, v in row.items():
                if v == "" or v is None:
                    entry[k] = None
                else:
                    try:
                        entry[k] = float(v)
                    except ValueError:
                        entry[k] = v
            history.append(entry)
    
    return history


def plot_all_curves(history, output_dir):
    """Génère toutes les courbes"""
    
    if not history:
        print("⚠️ Pas de données dans le CSV")
        return
    
    steps = [h["step"] for h in history]
    
    # ===== 1. LOSS =====
    fig, ax = plt.subplots(figsize=(12, 6))
    
    train_losses = [(h["step"], h["train_loss"]) for h in history if h.get("train_loss") is not None]
    val_losses = [(h["step"], h["val_loss"]) for h in history if h.get("val_loss") is not None]
    
    if train_losses:
        ax.plot([t[0] for t in train_losses], [t[1] for t in train_losses],
                marker='o', label='Train Loss', color='#2E86AB', linewidth=2)
    if val_losses:
        ax.plot([v[0] for v in val_losses], [v[1] for v in val_losses],
                marker='s', label='Val Loss', color='#E63946', linewidth=2)
    
    ax.set_xlabel('Step', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    loss_path = output_dir / "curves_loss.png"
    plt.savefig(loss_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"✅ {loss_path}")
    
    # ===== 2. F1 + PRECISION + RECALL =====
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # F1 train vs val
    train_f1 = [h.get("train_f1") for h in history]
    val_f1 = [h.get("val_f1") for h in history]
    
    axes[0].plot(steps, train_f1, marker='o', label='Train F1', color='#06A77D', linewidth=2)
    axes[0].plot(steps, val_f1, marker='s', label='Val F1', color='#D62246', linewidth=2)
    axes[0].set_xlabel('Step', fontsize=12)
    axes[0].set_ylabel('F1 Score', fontsize=12)
    axes[0].set_title('Train vs Val F1 Score', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11, loc='best')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(0, 1)
    
    # Highlight le meilleur F1
    if val_f1:
        valid_f1 = [(i, f) for i, f in enumerate(val_f1) if f is not None]
        if valid_f1:
            best_idx, best_f1 = max(valid_f1, key=lambda x: x[1])
            best_step = steps[best_idx]
            axes[0].axvline(x=best_step, color='gold', linestyle='--', alpha=0.7,
                           label=f'Best Val F1 = {best_f1:.3f}')
            axes[0].legend(fontsize=11)
    
    # Precision/Recall val
    val_p = [h.get("val_precision") for h in history]
    val_r = [h.get("val_recall") for h in history]
    
    axes[1].plot(steps, val_p, marker='o', label='Val Precision', color='#F18F01', linewidth=2)
    axes[1].plot(steps, val_r, marker='s', label='Val Recall', color='#A23B72', linewidth=2)
    axes[1].plot(steps, val_f1, marker='^', label='Val F1', color='#2E86AB', linewidth=2.5)
    axes[1].set_xlabel('Step', fontsize=12)
    axes[1].set_ylabel('Score', fontsize=12)
    axes[1].set_title('Validation : Precision / Recall / F1', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11, loc='best')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 1)
    
    plt.tight_layout()
    f1_path = output_dir / "curves_f1.png"
    plt.savefig(f1_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"✅ {f1_path}")
    
    # ===== 3. COURBE COMBINÉE (Dashboard) =====
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Top-left : Loss
    if train_losses:
        axes[0][0].plot([t[0] for t in train_losses], [t[1] for t in train_losses],
                       marker='o', label='Train', color='#2E86AB', linewidth=2)
    if val_losses:
        axes[0][0].plot([v[0] for v in val_losses], [v[1] for v in val_losses],
                       marker='s', label='Val', color='#E63946', linewidth=2)
    axes[0][0].set_title('Loss', fontsize=13, fontweight='bold')
    axes[0][0].set_xlabel('Step')
    axes[0][0].set_ylabel('Loss')
    axes[0][0].legend()
    axes[0][0].grid(True, alpha=0.3)
    
    # Top-right : F1
    axes[0][1].plot(steps, train_f1, marker='o', label='Train', color='#06A77D', linewidth=2)
    axes[0][1].plot(steps, val_f1, marker='s', label='Val', color='#D62246', linewidth=2)
    axes[0][1].set_title('F1 Score', fontsize=13, fontweight='bold')
    axes[0][1].set_xlabel('Step')
    axes[0][1].set_ylabel('F1')
    axes[0][1].set_ylim(0, 1)
    axes[0][1].legend()
    axes[0][1].grid(True, alpha=0.3)
    
    # Bottom-left : Precision val
    axes[1][0].plot(steps, val_p, marker='o', color='#F18F01', linewidth=2)
    axes[1][0].set_title('Val Precision', fontsize=13, fontweight='bold')
    axes[1][0].set_xlabel('Step')
    axes[1][0].set_ylabel('Precision')
    axes[1][0].set_ylim(0, 1)
    axes[1][0].grid(True, alpha=0.3)
    
    # Bottom-right : Recall val
    axes[1][1].plot(steps, val_r, marker='s', color='#A23B72', linewidth=2)
    axes[1][1].set_title('Val Recall', fontsize=13, fontweight='bold')
    axes[1][1].set_xlabel('Step')
    axes[1][1].set_ylabel('Recall')
    axes[1][1].set_ylim(0, 1)
    axes[1][1].grid(True, alpha=0.3)
    
    plt.suptitle('Training Dashboard - GLiNER CIR V2', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    
    dashboard_path = output_dir / "dashboard.png"
    plt.savefig(dashboard_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"✅ {dashboard_path}")
    
    # Stats
    print(f"\n📊 RÉSUMÉ")
    print("-" * 50)
    
    if val_f1:
        valid_f1 = [(i, f) for i, f in enumerate(val_f1) if f is not None]
        if valid_f1:
            best_idx, best_f1 = max(valid_f1, key=lambda x: x[1])
            print(f"  Meilleur Val F1   : {best_f1:.4f} (step {steps[best_idx]})")
            print(f"  Val F1 actuel     : {val_f1[-1]:.4f}")
    
    if val_losses:
        best_loss = min(v[1] for v in val_losses)
        print(f"  Meilleure Val Loss: {best_loss:.4f}")
        print(f"  Val Loss actuelle : {val_losses[-1][1]:.4f}")
    
    print(f"  Steps complétés   : {steps[-1] if steps else 0}")
    print(f"  Évaluations       : {len(history)}")


def main():
    print("=" * 60)
    print("VISUALISATION DES COURBES DE TRAINING")
    print("=" * 60)
    
    if not CSV_PATH.exists():
        print(f"\n❌ Pas de fichier de log : {CSV_PATH}")
        print(f"   Le training n'a peut-être pas encore commencé.")
        return
    
    history = load_history(CSV_PATH)
    
    if not history:
        print(f"\n⚠️ Le fichier est vide. Attends quelques évaluations.")
        return
    
    print(f"\n📂 {len(history)} points de mesure chargés")
    print(f"\n🎨 Génération des courbes...\n")
    
    plot_all_curves(history, MODEL_DIR)
    
    print(f"\n✅ Terminé !")
    print(f"\n📁 Fichiers dans : {MODEL_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
"""
====================================================================
ÉTAPE 4 : ÉVALUATION COMPLÈTE DU MODÈLE
====================================================================

Évalue ton nouveau GLiNER sur le test set :
- Métriques globales (precision, recall, F1)
- Métriques par label
- Matrice de confusion
- Comparaison avec l'ancien modèle (si disponible)
"""

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from gliner import GLiNER

BASE_DIR = Path(r"C:\EnnoSmart")
DATASET_DIR = BASE_DIR / "dataset_final"
MODEL_DIR = BASE_DIR / "models" / "gliner_cir_v2" / "final"
METRICS_DIR = BASE_DIR / "metrics_v2"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

LABELS_CORE = [
    "VERROU_TECH", "METHODE_RD", "TECHNOLOGIE_RD", "EQUIPEMENT_RD",
    "COMPOSANT_TECHNIQUE", "MATERIAU_SPECIFIQUE", "DOMAINE_RD",
    "RESULTAT_RD", "OBJECTIF_RD",
]

# Threshold à utiliser (sera optimisé sur val)
THRESHOLDS_TO_TEST = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]


def tokens_to_text(tokens):
    """Reconstruit le texte depuis les tokens"""
    return " ".join(tokens)


def evaluate_with_threshold(model, dataset, threshold):
    """Évalue le modèle avec un threshold donné"""
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)
    
    for item in dataset:
        text = tokens_to_text(item["tokenized_text"])
        gold_ner = item.get("ner", [])
        
        # Prédiction
        predictions = model.predict_entities(text, LABELS_CORE, threshold=threshold)
        
        # Convertir gold en (start_char, end_char, label) si possible
        # Pour simplifier, on compare en (label, text)
        gold_set = set()
        for ner_item in gold_ner:
            start_tok, end_tok, label = ner_item
            entity_text = " ".join(item["tokenized_text"][start_tok:end_tok+1])
            gold_set.add((label, entity_text.lower()))
        
        pred_set = set()
        for p in predictions:
            pred_set.add((p["label"], p["text"].lower()))
        
        # Compter par label
        for label in LABELS_CORE:
            gold_lbl = {(l, t) for l, t in gold_set if l == label}
            pred_lbl = {(l, t) for l, t in pred_set if l == label}
            
            tp[label] += len(gold_lbl & pred_lbl)
            fp[label] += len(pred_lbl - gold_lbl)
            fn[label] += len(gold_lbl - pred_lbl)
    
    # Calculer P, R, F1
    metrics = {}
    for label in LABELS_CORE:
        p = tp[label] / max(tp[label] + fp[label], 1)
        r = tp[label] / max(tp[label] + fn[label], 1)
        f1 = 2 * p * r / max(p + r, 1e-9)
        
        metrics[label] = {
            "precision": p,
            "recall": r,
            "f1": f1,
            "tp": tp[label],
            "fp": fp[label],
            "fn": fn[label],
        }
    
    # F1 micro global
    total_tp = sum(tp.values())
    total_fp = sum(fp.values())
    total_fn = sum(fn.values())
    
    micro_p = total_tp / max(total_tp + total_fp, 1)
    micro_r = total_tp / max(total_tp + total_fn, 1)
    micro_f1 = 2 * micro_p * micro_r / max(micro_p + micro_r, 1e-9)
    
    metrics["__global__"] = {
        "precision": micro_p,
        "recall": micro_r,
        "f1": micro_f1,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
    }
    
    return metrics


def find_best_threshold(model, val_data):
    """Cherche le meilleur threshold sur le val set"""
    print(f"\n🔍 Recherche du meilleur threshold sur VAL...")
    print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
    print("-" * 50)
    
    results = []
    for thr in THRESHOLDS_TO_TEST:
        metrics = evaluate_with_threshold(model, val_data, thr)
        g = metrics["__global__"]
        results.append({
            "threshold": thr,
            "precision": g["precision"],
            "recall": g["recall"],
            "f1": g["f1"],
        })
        print(f"  {thr:<10.2f} {g['precision']:<10.4f} {g['recall']:<10.4f} {g['f1']:<10.4f}")
    
    best = max(results, key=lambda x: x["f1"])
    print(f"\n🏆 Meilleur threshold : {best['threshold']} (F1 = {best['f1']:.4f})")
    
    return best["threshold"]


def main():
    print("=" * 70)
    print("ÉVALUATION DU MODÈLE GLINER V2")
    print("=" * 70)
    
    # Vérifier
    val_file = DATASET_DIR / "val.json"
    test_file = DATASET_DIR / "test.json"
    
    if not MODEL_DIR.exists():
        print(f"\n❌ Modèle introuvable : {MODEL_DIR}")
        print(f"   Lance d'abord : python 03_train_gliner.py")
        return
    
    if not val_file.exists() or not test_file.exists():
        print(f"\n❌ Datasets val/test introuvables")
        return
    
    # Charger
    print(f"\n📥 Chargement du modèle...")
    model = GLiNER.from_pretrained(str(MODEL_DIR))
    
    with open(val_file, "r", encoding="utf-8") as f:
        val_data = json.load(f)
    with open(test_file, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    
    print(f"   Val  : {len(val_data)} chunks")
    print(f"   Test : {len(test_data)} chunks")
    
    # 1. Trouver le meilleur threshold sur val
    best_threshold = find_best_threshold(model, val_data)
    
    # 2. Évaluer sur test avec le best threshold
    print(f"\n" + "=" * 70)
    print(f"📊 ÉVALUATION FINALE SUR TEST (threshold = {best_threshold})")
    print(f"=" * 70)
    
    test_metrics = evaluate_with_threshold(model, test_data, best_threshold)
    
    # Afficher
    global_m = test_metrics["__global__"]
    print(f"\n🎯 MÉTRIQUES GLOBALES (micro)")
    print(f"   Precision : {global_m['precision']:.4f}")
    print(f"   Recall    : {global_m['recall']:.4f}")
    print(f"   F1        : {global_m['f1']:.4f}")
    print(f"   TP/FP/FN  : {global_m['tp']}/{global_m['fp']}/{global_m['fn']}")
    
    print(f"\n📊 MÉTRIQUES PAR LABEL")
    print(f"{'Label':<22} {'Prec':<8} {'Recall':<8} {'F1':<8} {'TP':<6} {'FP':<6} {'FN':<6}")
    print("-" * 70)
    
    rows = []
    for label in LABELS_CORE:
        m = test_metrics[label]
        print(f"  {label:<20} {m['precision']:<7.3f} {m['recall']:<7.3f} "
              f"{m['f1']:<7.3f} {m['tp']:<5} {m['fp']:<5} {m['fn']:<5}")
        rows.append({"label": label, **m})
    
    # Sauvegarder
    metrics_file = METRICS_DIR / "test_metrics.json"
    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump({
            "best_threshold": best_threshold,
            "global": global_m,
            "per_label": {label: test_metrics[label] for label in LABELS_CORE},
        }, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 Métriques sauvegardées : {metrics_file}")
    
    # CSV pour visualisation
    df = pd.DataFrame(rows)
    csv_file = METRICS_DIR / "test_metrics.csv"
    df.to_csv(csv_file, index=False)
    print(f"   CSV : {csv_file}")
    
    # Comparaison avec l'ancien modèle (si dispo)
    print(f"\n" + "=" * 70)
    print(f"🎯 VERDICT FINAL")
    print(f"=" * 70)
    
    if global_m["f1"] >= 0.75:
        print(f"✅ EXCELLENT ! F1 = {global_m['f1']:.3f} (vs 0.36 ancien)")
        print(f"   Modèle prêt pour la production.")
    elif global_m["f1"] >= 0.60:
        print(f"✅ BON ! F1 = {global_m['f1']:.3f}")
        print(f"   Améliorable mais utilisable.")
    elif global_m["f1"] >= 0.45:
        print(f"⚠️ MOYEN. F1 = {global_m['f1']:.3f}")
        print(f"   Mieux que l'ancien (0.36) mais peut faire mieux.")
    else:
        print(f"❌ FAIBLE. F1 = {global_m['f1']:.3f}")
        print(f"   Quelque chose ne va pas, vérifier le training.")


if __name__ == "__main__":
    main()